<a href="https://colab.research.google.com/github/Hetic-Hackathon-Givaudan-2026-CHARK/equipe10/blob/main/notebooks/RAG_Givaudan_Equipe10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG — plateforme B2B d'arômes et de fragrances (équipe 10)

Ce notebook déroule toute la démo sur Colab :

1. installation (dépôt, Python, Ollama, modèles) ;
2. génération du dataset synthétique ;
3. évaluation de la **V2** sur les 30 cas de référence avec deux modèles d'embeddings (`nomic-embed-text` puis `qwen3-embedding:4b`) ;
4. lancement de l'API sur la V2 et exposition via ngrok.

**Avant de lancer** : choisir un environnement d'exécution avec GPU (*Exécution → Modifier le type d'exécution → GPU T4*) : sur CPU, les 80 réponses du LLM prennent beaucoup plus longtemps. Puis ajoute ton token ngrok dans les *Secrets* Colab (icône 🔑 à gauche) sous le nom `NGROK_AUTHTOKEN`. Ne jamais l'écrire en clair dans une cellule.

## 1. Installation

In [ ]:
BRANCH = "main"
!git clone -b $BRANCH https://github.com/Hetic-Hackathon-Givaudan-2026-CHARK/equipe10.git
%cd /content/equipe10
!pip install -q -r requirements.txt pyngrok

In [ ]:
# Ollama (serveur de modèles local)
!apt-get install -y -qq zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > /tmp/ollama.log 2>&1 &
import time; time.sleep(5)
!curl -s http://127.0.0.1:11434/api/tags

In [ ]:
!ollama pull nomic-embed-text
!ollama pull qwen3-embedding:4b
!ollama pull qwen2.5:1.5b
!ollama list

## 2. Dataset

In [ ]:
!python generate_dataset.py
!head -c 1500 givaudan_data.txt

In [ ]:
# Tests automatiques (données, ingestion, API, reclassement) — sans Ollama
!python -m pytest -q

## 3. Évaluation de la V2 sur les 30 cas de référence

Protocole et seuils : `docs/EVALUATION.md`. Il faut réindexer à chaque changement de modèle d'embeddings.

In [ ]:
# Baseline lexicale sans IA (TF-IDF), avec et sans reclassement : ne nécessite pas Ollama
!python baseline_tfidf.py --label baseline_tfidf | tail -6
!python baseline_tfidf.py --rerank --label baseline_tfidf_rerank | tail -6
!python baseline_tfidf.py --rerank --cases data/eval/unseen_cases.json --label baseline_tfidf_rerank_inedits | tail -6

In [ ]:
%env EMBED_MODEL=nomic-embed-text
%env LLM_MODEL=qwen2.5:1.5b
!python clean.py && python ingest.py
!python evaluate.py --label v2_nomic
!python evaluate.py --label v2_nomic_inedits --cases data/eval/unseen_cases.json

In [ ]:
%env EMBED_MODEL=qwen3-embedding:4b
!python clean.py && python ingest.py
!python evaluate.py --label v2_qwen3
!python evaluate.py --label v2_qwen3_inedits --cases data/eval/unseen_cases.json

In [ ]:
# Synthèse de toutes les configurations (baseline, nomic, qwen3, cas inédits)
!python report.py

In [ ]:
# Télécharger tous les rapports d'évaluation (à pousser dans results/ sur GitHub)
!zip -qr results.zip results
from google.colab import files
files.download("results.zip")

## 4. API (V2) et exposition via ngrok

In [ ]:
# L'index ChromaDB est celui de qwen3-embedding:4b (dernière ingestion) ; l'API hérite de EMBED_MODEL.
!nohup python api.py > /tmp/fastapi.log 2>&1 &
time.sleep(8)
!curl -s http://127.0.0.1:8000/health

In [ ]:
!curl -s -X POST http://127.0.0.1:8000/ask -H "Content-Type: application/json" \
  -d '{"question": "Je cherche un arôme mangue naturel et vegan pour une boisson protéinée."}'

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
ngrok.kill()
public_url = ngrok.connect(8000).public_url
print("Swagger :", f"{public_url}/docs")
print("Ping    :", f"{public_url}/demo/ping")